In [1]:
# Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

# Import the libraries we need.
import os
import numpy as np
import pandas as pd

from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Define the project folder.
project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

# Define the dataset paths.
data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'

# Define the anomaly label file.
labels_path = project_path + '/data/raw/archive/labeled_anomalies.csv'

print("SERIAL 13 loaded successfully.")
print("Project folder exists:", os.path.exists(project_path))
print("Labels file exists:", os.path.exists(labels_path))

Mounted at /content/drive
SERIAL 13 loaded successfully.
Project folder exists: True
Labels file exists: True


#A-8 test telemetry and its ground-truth anomaly information from SERIAL 12.

In [2]:
# Select the spacecraft telemetry channel.
channel = "A-8"

# Load the A-8 test telemetry data.
data_A8 = np.load(test_path + "/" + channel + ".npy")

# The first column contains the main telemetry signal.
telemetry_A8 = data_A8[:, 0]

# Load the ground-truth anomaly labels.
labels_df = pd.read_csv(labels_path)

# Find the label information for A-8.
A8_info = labels_df[labels_df["chan_id"] == channel].iloc[0]

print("Channel:", channel)
print("Test data shape:", data_A8.shape)
print("Number of telemetry points:", len(telemetry_A8))
print("Ground-truth anomaly sequence:", A8_info["anomaly_sequences"])
print("Anomaly class:", A8_info["class"])

Channel: A-8
Test data shape: (8375, 25)
Number of telemetry points: 8375
Ground-truth anomaly sequence: [[4569, 8374]]
Anomaly class: [contextual]


In [3]:
# Create ground-truth labels for every test time step.
# Start by assuming every point is normal.
y_true_A8 = np.zeros(len(telemetry_A8), dtype=int)

# Get the anomaly start and end positions.
anomaly_start = 4569
anomaly_end = 8374

# Mark the known anomaly interval as 1.
y_true_A8[anomaly_start:anomaly_end + 1] = 1

# Count normal and anomaly points.
normal_count = np.sum(y_true_A8 == 0)
anomaly_count = np.sum(y_true_A8 == 1)

print("Normal points:", normal_count)
print("Anomaly points:", anomaly_count)
print("Total points:", len(y_true_A8))

Normal points: 4569
Anomaly points: 3806
Total points: 8375


In [4]:
# Load the A-8 training telemetry.
train_A8 = np.load(train_path + "/" + channel + ".npy")
train_telemetry_A8 = train_A8[:, 0]

# Calculate the normal mean and standard deviation from training data.
normal_mean = np.mean(train_telemetry_A8)
normal_std = np.std(train_telemetry_A8)

# Calculate Z-scores for the A-8 test telemetry.
z_scores_A8 = (telemetry_A8 - normal_mean) / normal_std

# Apply the global ±3 threshold.
# A point is predicted as an anomaly when its absolute Z-score is greater than 3.
y_pred_zscore_A8 = (np.abs(z_scores_A8) > 3).astype(int)

# Count the predictions.
predicted_normal = np.sum(y_pred_zscore_A8 == 0)
predicted_anomaly = np.sum(y_pred_zscore_A8 == 1)

print("Predicted normal points:", predicted_normal)
print("Predicted anomaly points:", predicted_anomaly)

Predicted normal points: 8375
Predicted anomaly points: 0


In [5]:
# Calculate evaluation metrics for the Global Z-score method.
precision = precision_score(y_true_A8, y_pred_zscore_A8, zero_division=0)
recall = recall_score(y_true_A8, y_pred_zscore_A8, zero_division=0)
f1 = f1_score(y_true_A8, y_pred_zscore_A8, zero_division=0)

print("Global Z-score Evaluation")
print("-------------------------")
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Global Z-score Evaluation
-------------------------
Precision: 0.0
Recall: 0.0
F1-score: 0.0


In [6]:
# Calculate the confusion matrix.
cm = confusion_matrix(y_true_A8, y_pred_zscore_A8)

# Extract the four values.
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix")
print("----------------")
print("True Negatives (TN):", tn)
print("False Positives (FP):", fp)
print("False Negatives (FN):", fn)
print("True Positives (TP):", tp)

Confusion Matrix
----------------
True Negatives (TN): 4569
False Positives (FP): 0
False Negatives (FN): 3806
True Positives (TP): 0


In [7]:
# Calculate the false positive and false negative rates.

false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)

print("False Positive Rate:", false_positive_rate)
print("False Negative Rate:", false_negative_rate)

False Positive Rate: 0.0
False Negative Rate: 1.0


#Save the Evaluation Results

In [8]:
# Create a summary of the Global Z-score evaluation.
evaluation_summary = pd.DataFrame({
    "channel": [channel],
    "method": ["Global Z-score"],
    "threshold": ["±3"],
    "precision": [precision],
    "recall": [recall],
    "f1_score": [f1],
    "false_positive_rate": [false_positive_rate],
    "false_negative_rate": [false_negative_rate],
    "true_negatives": [tn],
    "false_positives": [fp],
    "false_negatives": [fn],
    "true_positives": [tp]
})

# Define the results folder.
results_path = project_path + "/results"

# Create the folder if it does not already exist.
os.makedirs(results_path, exist_ok=True)

# Save the evaluation results.
output_file = results_path + "/global_zscore_evaluation_A8.csv"
evaluation_summary.to_csv(output_file, index=False)

print("Evaluation results saved.")
print("File:", output_file)
print("\n")
print(evaluation_summary)

Evaluation results saved.
File: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/global_zscore_evaluation_A8.csv


  channel          method threshold  precision  recall  f1_score  \
0     A-8  Global Z-score        ±3        0.0     0.0       0.0   

   false_positive_rate  false_negative_rate  true_negatives  false_positives  \
0                  0.0                  1.0            4569                0   

   false_negatives  true_positives  
0             3806               0  


The Global Z-score method was not generating false alarms, but it completely failed to detect the known A-8 contextual anomaly.

This gives us a strong baseline for comparing the next methods.

#github commit

In [9]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [10]:
# Go to the project folder that contains the Git repository.
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

# Check the Git repository status.
!git status

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (17/17), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/12_statistical_anomaly_detection.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/13_evaluate_global_zscore.ipynb
	results/global_zscore_evaluation_A8.csv

no changes added to commit (use "git add" and/or "git commit -a")
